# Show Image Examples

Draw a few random YFCC4k images (or pin specific ids) and run **every attack of the paper**
against each one, then save the clean and attacked images to `.media/images_examples/`.
These are the qualitative examples / teaser figures used in the paper.

The attacks are the *standard* specifications of the `cluster_full` run
(`configs/cluster_full/config.yaml`, see `configs/cluster_full/ATTACK_SPECS.md`): pixel-space
perturbations (a nominal 8/255 really is 8 grey levels), the true location fed to every
attack (`true_gps`), and restarts selected by the FSD to the true location. Paper-facing names
come from `utils.plots.common._display_attack_name` -- every attack except GeoShield is an
instance of the **GeoFA** framework:

| Key (`run_attack`) | Paper name | Notes |
|--------------------|------------|-------|
| `dtd` | GeoFA-TD (ours) | Trajectory deviation (cosine) from the clean velocity |
| `diffusion_l2` | GeoFA-L² | Same forward process, L2 metric |
| `targeted_l2` | GeoFA-DPA | RFM training loss minimised towards the antipode |
| `sampling` | GeoFA-sample | Haversine to the truth through the 16-step sampler (w=0) |
| `encoder` | GeoFA-enc. | Encoder-space attack (sign-SGD PGD) |
| `training_loss` | GeoFA-AdvDM | Maximise the model's own training loss |
| `unidef` | GeoFA-UniDef | AdvDM objective with the FDJE gradient estimator |
| `geoshield` | GeoShield | *(out-of-process)* external Geoshield repo, model-exact 336px PNGs |

Output layout (one folder per image; the primary budget, 8/255, sits at the folder root so
the paper's asset paths stay stable, every other budget gets a `budget_<n>_255/` subfolder
with the same content):

```
.media/images_examples/
  <image_id>/                       # PRIMARY_BUDGET (8/255)
    original.png          # full-resolution clean image
    clean.png             # model-preprocessed clean image (336x336; the space every attack lives in)
    dtd.png  diffusion_l2.png  targeted_l2.png  sampling.png  encoder.png
    training_loss.png  unidef.png  geoshield.png
    comparison_grid.png   # side-by-side montage
    prediction_points.json, prediction_map.png / prediction_map_max.pdf
    tikz/                 # standalone pgfplots teaser
    budget_1_255/                   # same files for the 1/255 budget
      clean.png  dtd.png ...  prediction_map_max.pdf  tikz/
  metadata.json
```

## Setup

In [ ]:
import os
# Run from adversarial_demo/ (parent of demo_notebooks/). Idempotent.
if os.path.basename(os.getcwd()) == "demo_notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

cwd: /users/eleves-b/2023/mathias.ollu/repos/plonk/adversarial_demo


In [ ]:
import json
import random
import shutil
import sys
from pathlib import Path

import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from attacks.attacks import run_attack
from utils.pipe_trajectory import PlonkPipelineTrajectory
from utils.adversarial_utils import (
    add_perturbation_to_image,
    conditional_preprocessing,
    tensor_to_pil,
    seed_everything,
    resolve_torch_device,
)
from utils.datasets import retrieve_yfcc_images
from utils.plots.common import _display_attack_name

print("Imports OK")

## Configuration

Pick which images to use, the L-inf budgets for the figures, and which attacks to run.
Either pin explicit YFCC ids in `EXAMPLE_IMAGE_IDS` (the paper's examples) or set it to
`None` to draw `N_EXAMPLE_IMAGES` random ones (`SELECTION_SEED` controls the draw).
`ATTACK_SEED` seeds the attack optimisation (matches the `seed` in the config).

In [ ]:
DATASET = "yfcc"
# Pin the paper's example image(s) here; set to None to draw random ones instead.
EXAMPLE_IMAGE_IDS = ["11016735633"]
N_EXAMPLE_IMAGES = 3          # number of random images to draw (when EXAMPLE_IMAGE_IDS is None)
SELECTION_SEED = 128          # seed for drawing the random YFCC images (change to resample)
POOL_SIZE = 200               # draw the images from this many seeded YFCC samples
ATTACK_SEED = 42              # seed for the attack optimisation (matches config.yaml)
# L-inf budgets (pixel scale) for the qualitative figures. PRIMARY_BUDGET's outputs go to the
# image folder root, every other budget to a `budget_<n>_255/` subfolder.
ATTACK_BUDGETS = [8 / 255, 1 / 255]
PRIMARY_BUDGET = 8 / 255
# Resumable: skip an (image, budget, attack) whose PNG already exists (delete it to redo).
SKIP_EXISTING = True

# In-process attacks, canonical `run_attack` keys (= file stems, = keys of _display_attack_name).
# The paper's set: the cluster_full run minus its ablation variants (sampling_cfg10, dtd_blur).
ATTACKS = [
    "dtd",            # GeoFA-TD (ours)
    "diffusion_l2",   # GeoFA-L2
    "targeted_l2",    # GeoFA-DPA
    "sampling",       # GeoFA-sample
    "encoder",        # GeoFA-enc.
    "training_loss",  # GeoFA-AdvDM
    "unidef",         # GeoFA-UniDef
]

RUN_GEOSHIELD = True          # also run the out-of-process GeoShield attack (Geoshield repo)

# Fast dry-run (lower fidelity, NOT for the paper figure): set True to sanity-check quickly.
FAST_MODE = False

# Paper-figure assets live under the repo's .media/ (one folder per image).
OUTPUT_DIR = Path("../.media/images_examples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve())
print("Attacks:", {k: _display_attack_name(k) for k in ATTACKS + (["geoshield"] if RUN_GEOSHIELD else [])})

Machine-specific paths (data dirs, pipelines) come from the local `config.yaml`; the attack
hyper-parameters come from the paper run's `configs/cluster_full/config.yaml` (the
`attack_train_args` block: 60 sign-SGD steps, 5 restarts kept by FSD to the true location,
...). We build the shared kwargs exactly like `main.get_attack_kwargs` does: the per-dataset
block minus the `per_attack` overrides, which are applied per attack type below. The `eps_max`
budget is added per figure by `attack_kwargs_for(budget)`.

In [ ]:
with open("config.yaml") as f:
    config = yaml.safe_load(f)            # local paths / pipelines
with open("configs/cluster_full/config.yaml") as f:
    paper_config = yaml.safe_load(f)      # the paper run's attack protocol

device = resolve_torch_device(config.get("device", "cuda"))

base_kwargs = dict(paper_config["attack_train_args"][DATASET])
per_attack_overrides = base_kwargs.pop("per_attack", {}) or {}
base_kwargs["device"] = device
_rev = base_kwargs.get("restart_eval_seed")
base_kwargs["restart_eval_seed"] = ATTACK_SEED if _rev in (None, "seed") else int(_rev)

if FAST_MODE:
    base_kwargs.update(
        n_steps=20, num_restarts=1, restart_eval_num_steps=16,
        anchor_samples=128, restart_eval_batch_size=64,
    )

def budget_label(budget):
    """'8_255' style tag for folder names."""
    return f"{round(budget * 255)}_255"

def attack_kwargs_for(budget):
    return {**base_kwargs, "eps_max": float(budget)}

print("Device:", device)
print("Budgets:", [f"{round(b * 255)}/255" for b in ATTACK_BUDGETS], "(primary:", f"{round(PRIMARY_BUDGET * 255)}/255)")
print("Base attack kwargs:", base_kwargs)
print("Per-attack overrides:", per_attack_overrides)

In [5]:
print("Loading pipeline:", config["pipelines"][DATASET])
pipeline = PlonkPipelineTrajectory(config["pipelines"][DATASET]).to(device)
print("Pipeline loaded")

Loading pipeline: nicolas-dufour/PLONK_YFCC


Using cache found in /users/eleves-b/2023/mathias.ollu/.cache/torch/hub/facebookresearch_dinov2_main
/users/eleves-b/2023/mathias.ollu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/users/eleves-b/2023/mathias.ollu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/users/eleves-b/2023/mathias.ollu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Pipeline loaded


## Select the images

Either the pinned `EXAMPLE_IMAGE_IDS` (loaded by id with their ground-truth GPS) or a
seeded random draw from YFCC4k.

In [ ]:
if EXAMPLE_IMAGE_IDS:
    example_images, example_gps, example_ids = [], [], []
    for iid in EXAMPLE_IMAGE_IDS:
        imgs, gps, ids = retrieve_yfcc_images(im_idx=iid, local_dir=config["data_dirs"][DATASET])
        example_images += imgs; example_gps += gps; example_ids += ids
else:
    # Draw N_EXAMPLE_IMAGES random images from a seeded YFCC4k pool (reproducible).
    pool_imgs, pool_gps, pool_ids = retrieve_yfcc_images(
        n_images_to_eval=POOL_SIZE, seed=SELECTION_SEED, local_dir=config["data_dirs"][DATASET],
    )
    _rng = random.Random(SELECTION_SEED)
    _idx = _rng.sample(range(len(pool_imgs)), N_EXAMPLE_IMAGES)
    example_images = [pool_imgs[i] for i in _idx]
    example_gps = [pool_gps[i] for i in _idx]
    example_ids = [pool_ids[i] for i in _idx]
print("Selected image ids:", example_ids)

fig, axes = plt.subplots(1, len(example_ids), figsize=(4 * len(example_ids), 4))
for ax, img, iid, gps in zip(np.atleast_1d(axes), example_images, example_ids, example_gps):
    ax.imshow(img)
    ax.set_title(f"{iid}\n({gps[0]:.2f}, {gps[1]:.2f})")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def image_dir(image_id):
    d = OUTPUT_DIR / str(image_id)
    d.mkdir(parents=True, exist_ok=True)
    return d

def budget_dir(image_id, budget):
    """Folder holding one image's outputs at one budget: the image folder itself for
    PRIMARY_BUDGET, a `budget_<n>_255/` subfolder otherwise."""
    d = image_dir(image_id)
    if abs(budget - PRIMARY_BUDGET) > 1e-9:
        d = d / f"budget_{budget_label(budget)}"
        d.mkdir(parents=True, exist_ok=True)
    return d

# Save clean images: full-resolution original + model-preprocessed (same space as attacks),
# then a copy of clean.png into every budget folder so each one is self-contained.
for img, iid in zip(example_images, example_ids):
    d = image_dir(iid)
    img.save(d / "original.png")
    clean_tensor = conditional_preprocessing(img, pipeline, device=device)
    tensor_to_pil(clean_tensor, pipeline).save(d / "clean.png")
    for budget in ATTACK_BUDGETS:
        bd = budget_dir(iid, budget)
        if bd != d:
            shutil.copy(d / "clean.png", bd / "clean.png")
print("Saved clean images.")

## Run the in-process attacks (every budget)

Each attack is trained per image with the shared config kwargs (plus its per-attack
overrides) and the image's **true GPS** (`true_gps`: the standard GeoFA attacks noise the
true location in their forward process, GeoFA-DPA aims at its antipode, and the restart
selection keeps the restart with the highest FSD to the truth). The selected perturbation
`delta` (pixel units) is applied to the clean image, quantised to 8 bit and saved.

In [ ]:
records = []

def run_one_attack(attack_type, source_image, true_gps, budget):
    kwargs = attack_kwargs_for(budget)
    kwargs.update(per_attack_overrides.get(attack_type, {}))
    kwargs["true_gps"] = tuple(float(x) for x in true_gps)
    seed_everything(ATTACK_SEED)
    result = run_attack(
        attack_type=attack_type,
        source_image=source_image,
        pipeline=pipeline,
        silent=True,
        **kwargs,
    )
    attacked = add_perturbation_to_image(source_image, result["delta"], pipeline)
    metrics = result.get("best_metrics") or {}
    disp = metrics.get("final_step_displacement_true", metrics.get("final_step_displacement"))
    return attacked, disp

for budget in ATTACK_BUDGETS:
    eps_tag = f"{round(budget * 255)}/255"
    for img, iid, gps in zip(example_images, example_ids, example_gps):
        d = budget_dir(iid, budget)
        for atype in ATTACKS:
            out = d / f"{atype}.png"
            if SKIP_EXISTING and out.exists():
                print(f"[{iid} @ {eps_tag}] {atype}: exists, skipping", flush=True)
                records.append({"image_id": iid, "budget_over_255": round(budget * 255),
                                "attack": atype, "final_step_displacement_km": None, "skipped": True})
                continue
            print(f"[{iid} @ {eps_tag}] running {atype} ({_display_attack_name(atype)}) ...", flush=True)
            attacked, disp = run_one_attack(atype, img, gps, budget)
            attacked.save(out)
            records.append({"image_id": iid, "budget_over_255": round(budget * 255),
                            "attack": atype, "final_step_displacement_km": disp})
            msg = f"   saved {out.relative_to(OUTPUT_DIR)}"
            if disp is not None:
                msg += f"  (FSD to true location {disp:.1f} km)"
            print(msg, flush=True)

print("In-process attacks done.")

## GeoShield (out-of-process)

GeoShield is not a trainable attack in this repo: its perturbations are produced by the
separate **Geoshield** repo (`Geoshield/geoshield.py`). Following the `cluster_full`
protocol, we hand it **lossless PNG copies of the model-preprocessed 336x336 square**
(`clean.png`, exactly the clean tensor the trainable attacks are evaluated on) and run it at
`input_res=336`, so its L-inf budget (`round(budget * 255)` grey levels) is applied at the
evaluated resolution and its crop/resize is a no-op. The attacked PNGs are copied back.
Wrapped in `try/except` so the notebook still completes if the GeoShield environment is
unavailable.

In [ ]:
if RUN_GEOSHIELD:
    from utils.geoshield import (
        _run_geoshield_generation,
        geoshield_epsilon_for_budget,
        default_repos_root,
    )

    gcfg = paper_config.get("geoshield", {}) or {}
    repos_root = gcfg.get("repos_root") or str(default_repos_root())

    for budget in ATTACK_BUDGETS:
        epsilon = geoshield_epsilon_for_budget(budget)
        todo = {iid: budget_dir(iid, budget) for iid in example_ids
                if not (SKIP_EXISTING and (budget_dir(iid, budget) / "geoshield.png").exists())}
        if not todo:
            print(f"GeoShield @ {epsilon}/255: all outputs exist, skipping")
            continue

        # Absolute, per-epsilon paths: GeoShield runs from repos_root, so relative paths would
        # resolve against the wrong directory, and its output dir is keyed by config hash.
        gs_clean = (OUTPUT_DIR / "_geoshield_clean" / f"eps{epsilon}").resolve()
        gs_out = (OUTPUT_DIR / "_geoshield_out" / f"eps{epsilon}").resolve()
        if gs_clean.exists():
            shutil.rmtree(gs_clean)
        gs_clean.mkdir(parents=True)

        # Model-exact clean copies: clean.png is the pipeline's own preprocessed square (PNG, lossless).
        name_to_id = {}
        input_res = None
        for iid in todo:
            fname = f"{iid}.png"
            clean_png = image_dir(iid) / "clean.png"
            shutil.copy(clean_png, gs_clean / fname)
            with Image.open(clean_png) as im:
                assert im.size[0] == im.size[1], f"clean.png is not square: {im.size}"
                input_res = im.size[0]
            name_to_id[fname] = iid

        try:
            attacked_dir = _run_geoshield_generation(
                repos_root=repos_root,
                script=gcfg.get("script", "Geoshield/geoshield.py"),
                python_exe=gcfg.get("python") or sys.executable,
                clean_dir=str(gs_clean),
                output_path=str(gs_out),
                n_images=len(name_to_id),
                epsilon=int(epsilon),
                steps=int(gcfg.get("steps", 100)),
                input_res=int(gcfg.get("input_res") or input_res),
            )
            for fname, iid in name_to_id.items():
                stem = Path(fname).stem
                cands = list(Path(attacked_dir).glob(stem + ".*"))
                if cands:
                    Image.open(cands[0]).convert("RGB").save(todo[iid] / "geoshield.png")
                    records.append({"image_id": iid, "budget_over_255": round(budget * 255),
                                    "attack": "geoshield", "final_step_displacement_km": None})
                    print(f"[{iid} @ {epsilon}/255] saved geoshield.png")
                else:
                    print(f"[{iid} @ {epsilon}/255] WARNING: no GeoShield output found for {stem}")
        except Exception as exc:
            print(f"GeoShield generation failed @ {epsilon}/255 (skipping):", exc)
else:
    print("RUN_GEOSHIELD is False; skipping GeoShield.")

## Comparison grids

In [ ]:
panel_order = ["clean"] + list(ATTACKS) + (["geoshield"] if RUN_GEOSHIELD else [])

for budget in ATTACK_BUDGETS:
    for iid in example_ids:
        d = budget_dir(iid, budget)
        panels = [(name, d / f"{name}.png") for name in panel_order if (d / f"{name}.png").exists()]
        n = len(panels)
        fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 2.8))
        for ax, (name, path) in zip(np.atleast_1d(axes), panels):
            ax.imshow(Image.open(path))
            ax.set_title("clean" if name == "clean" else _display_attack_name(name), fontsize=9)
            ax.axis("off")
        fig.suptitle(f"Image {iid} -- budget {round(budget * 255)}/255")
        plt.tight_layout()
        fig.savefig(d / "comparison_grid.png", dpi=150, bbox_inches="tight")
        plt.show()

## Map: predicted locations under attack

For each selected image and budget we plot, on a world map, the **true GPS location**, the **clean
prediction**, and the **predicted location of every attack** (including **GeoFA-TD**, this
paper's headline attack). PLONK is generative, so each image yields a *distribution* of GPS
samples; we summarise each with its **geometric medoid** (the sample minimising total
great-circle distance to the rest — robust to multi-modal spreads and the date line).

Design choices for a publication-ready, easy-to-read figure:

- **True location** = gold star, **clean prediction** = white circle, **GeoFA-TD (ours)** =
  bold red star, every other attack a distinct colour/marker (legend names are the paper's,
  via `_display_attack_name`).
- Faint great-circle **connectors** run from the clean prediction to each attack, so the
  displacement each attack induces is visible at a glance (GeoFA-TD's is drawn thicker).
- A **zoom inset** frames the region around the true location. Attacks that fail to move
  the prediction land *on top of* the clean prediction; in the inset these coincident
  markers are **exploded onto a small ring** (with thin grey leader lines) so each stays
  individually visible.

Predictions are cached to `prediction_points.json` per image, so you can restyle the map
without re-running the pipeline (set `RECOMPUTE_PREDICTIONS = True` to force a refresh;
the cache is also refreshed automatically when an attack image is newer than it).

In [ ]:
import matplotlib.lines as mlines
from matplotlib.transforms import Bbox
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from utils.adversarial_utils import make_shared_initial_noise

# --- map sampling settings ---
MAP_NUM_STEPS = 250           # sampling steps for the predicted-GPS distribution
MAP_N_SAMPLES = 256           # number of GPS samples drawn per image
MAP_SEED = 1234               # shared initial noise / seed (clean & attacks comparable)
RECOMPUTE_PREDICTIONS = False  # True => ignore cached prediction_points.json and re-sample
GLOBAL_FAN_SPAN = 9.0         # degrees; explode coincident attack clusters on the world map

# The headline attack (this paper's contribution) gets the highlighted marker treatment
# (black edge, larger size) on the map/legend and the "(ours)" suffix, like the TikZ teaser.
OURS_ATTACKS = ("dtd",)

def attack_legend_name(key):
    name = _display_attack_name(key)
    return f"{name} (ours)" if key in OURS_ATTACKS else name

# attack key -> (colour, marker, size). Legend names come from _display_attack_name.
ATTACK_STYLE = {
    "dtd":           ("#e6194B", "*", 430),
    "diffusion_l2":  ("#42d4f4", "X", 130),
    "targeted_l2":   ("#911eb4", "v", 95),
    "sampling":      ("#808080", "h", 95),
    "encoder":       ("#4363d8", "s", 90),
    "training_loss": ("#f58231", "P", 100),
    "unidef":        ("#3cb44b", "^", 95),
    "geoshield":     ("#bfef45", "p", 100),
}
ATTACK_ORDER = list(ATTACK_STYLE.keys())


def haversine_km(a, b):
    """Great-circle distance (km) from point a=(lat,lon) to each row of b=[N,2] (lat,lon)."""
    R = 6371.0
    b = np.atleast_2d(b)
    lat1, lon1, lat2, lon2 = map(np.radians, [a[0], a[1], b[:, 0], b[:, 1]])
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(h))


def medoid(gps):
    """gps: [N,2] (lat,lon). Returns the sample minimising total haversine to the rest."""
    best, best_cost = 0, np.inf
    for i in range(len(gps)):
        c = float(haversine_km((gps[i, 0], gps[i, 1]), gps).sum())
        if c < best_cost:
            best_cost, best = c, i
    return gps[best]


@torch.inference_mode()
def predict_gps(image):
    """Sample MAP_N_SAMPLES predicted GPS points for a PIL image (shared noise + seed)."""
    import random
    x_n = make_shared_initial_noise(MAP_N_SAMPLES, device=device, seed=MAP_SEED)
    random.seed(MAP_SEED); np.random.seed(MAP_SEED); torch.manual_seed(MAP_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MAP_SEED)
    gps, _ = pipeline(image, batch_size=MAP_N_SAMPLES, cfg=10.0, x_N=x_n,
                      return_trajectories=True, num_steps=MAP_NUM_STEPS)
    return gps.detach().cpu().numpy() if torch.is_tensor(gps) else np.asarray(gps)


def _clusters(coords, thresh_km):
    """Connected components of points closer than thresh_km (union-find)."""
    n = len(coords)
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for i in range(n):
        for j in range(i + 1, n):
            if haversine_km(tuple(coords[i]), coords[j])[0] < thresh_km:
                parent[find(i)] = find(j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())


def fan_layout(preds, clean_pred, span_deg, explode_clean=True):
    """Explode markers on (nearly) the same spot onto a small ring so each stays visible.

    Returns {label: (disp_lat, disp_lon, anchor_or_None)}; anchor is the real shared
    location a grey leader line is drawn back to (None when the marker was not moved).
    A single attack sitting on the clean prediction (e.g. GeoShield failing to move it) is
    exploded too, otherwise the clean circle (drawn on top) would hide it entirely.
    With explode_clean=False, clusters sitting on the clean prediction are left stacked
    (the zoom inset reveals those) so the global map's origin stays uncluttered."""
    labels = list(preds.keys())
    coords = [np.asarray(preds[l], dtype=float) for l in labels]
    thresh = max(150.0, span_deg * 35)
    r = span_deg * 0.16
    layout = {}
    for grp in _clusters(coords, thresh):
        cen = np.mean([coords[i] for i in grp], axis=0)
        near_clean = haversine_km(tuple(clean_pred), cen)[0] < thresh
        if (len(grp) == 1 and not near_clean) or (near_clean and not explode_clean):
            for i in grp:
                layout[labels[i]] = (coords[i][0], coords[i][1], None)
            continue
        anchor = np.asarray(clean_pred, dtype=float) if near_clean else cen
        for k, i in enumerate(grp):
            ang = 2 * np.pi * k / len(grp) + np.pi / 2
            dlat = anchor[0] + r * np.sin(ang)
            dlon = anchor[1] + r * np.cos(ang) / max(0.3, np.cos(np.radians(anchor[0])))
            layout[labels[i]] = (dlat, dlon, (anchor[0], anchor[1]))
    return layout


def _style_map(ax, labels=True):
    ax.add_feature(cfeature.OCEAN, facecolor="#eaf1f7")
    ax.add_feature(cfeature.LAND, facecolor="#ededed", edgecolor="none")
    ax.add_feature(cfeature.COASTLINE, edgecolor="#888888", linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="#aaaaaa", linewidth=0.4)
    gl = ax.gridlines(draw_labels=labels, linewidth=0.3, color="#bbbbbb", alpha=0.5, linestyle="--")
    gl.top_labels = False; gl.right_labels = False
    gl.xlabel_style = {"size": 7}; gl.ylabel_style = {"size": 7}


def _draw_points(ax, true_gps, clean_pred, preds, with_connectors=True, fan_span=None,
                 explode_clean=True):
    """Draw connectors + markers. With fan_span (deg) coincident markers are exploded."""
    layout = (fan_layout(preds, clean_pred, fan_span, explode_clean) if fan_span
              else {l: (preds[l][0], preds[l][1], None) for l in preds})
    if with_connectors:
        for label, pt in preds.items():
            ax.plot([clean_pred[1], pt[1]], [clean_pred[0], pt[0]], transform=ccrs.Geodetic(),
                    color=ATTACK_STYLE[label][0], linewidth=2.2 if label in OURS_ATTACKS else 0.9,
                    alpha=0.55, zorder=3)
    for label, (dlat, dlon, anchor) in layout.items():
        if anchor is not None:  # leader line for an exploded marker
            ax.plot([anchor[1], dlon], [anchor[0], dlat], transform=ccrs.PlateCarree(),
                    color="#555555", linewidth=0.7, alpha=0.8, zorder=6)
    for label in ATTACK_ORDER:
        if label not in layout:
            continue
        dlat, dlon, _ = layout[label]
        color, marker, size = ATTACK_STYLE[label]
        ax.scatter(dlon, dlat, color=color, marker=marker, s=size,
                   edgecolors="black" if label in OURS_ATTACKS else "white",
                   linewidths=1.6 if label in OURS_ATTACKS else 0.7,
                   zorder=11 if label in OURS_ATTACKS else 8, transform=ccrs.PlateCarree())
    ax.scatter(clean_pred[1], clean_pred[0], facecolor="white", marker="o", s=150,
               edgecolors="black", linewidths=1.3, zorder=9, transform=ccrs.PlateCarree())
    ax.scatter(true_gps[1], true_gps[0], color="#ffd700", marker="*", s=520,
               edgecolors="black", linewidths=1.4, zorder=12, transform=ccrs.PlateCarree())


def legend_handles(preds):
    h = [mlines.Line2D([], [], color="#ffd700", marker="*", linestyle="None",
                       markersize=18, markeredgecolor="black", label="True location"),
         mlines.Line2D([], [], color="white", marker="o", linestyle="None",
                       markersize=10, markeredgecolor="black", label="Clean prediction")]
    for label in ATTACK_ORDER:
        if label not in preds:
            continue
        color, marker, _ = ATTACK_STYLE[label]
        h.append(mlines.Line2D([], [], color=color, marker=marker, linestyle="None",
                               markersize=11 if label in OURS_ATTACKS else 8,
                               markeredgecolor="black" if label in OURS_ATTACKS else "white",
                               label=attack_legend_name(label)))
    return h


def perturbation_views(clean_img, attacked_img, pct=99.5):
    """Two complementary renderings of the GeoFA-TD perturbation delta = attacked - clean:

    - signed: delta centred at mid-grey and amplified (gain from a high percentile of
      |delta|, so outliers don't saturate it) so the additive perturbation is visible;
    - heatmap: per-pixel L2 magnitude |delta| on a perceptual colourmap, which reveals
      *where* the attack concentrates its perturbation (typically edges / salient content).
    Returns (signed_uint8, heatmap_uint8, gain)."""
    c = np.asarray(clean_img.convert("RGB"), dtype=float)
    a = np.asarray(attacked_img.convert("RGB"), dtype=float)
    diff = a - c
    scale = max(1.0, float(np.percentile(np.abs(diff), pct)))
    gain = 127.5 / scale
    signed = np.clip(127.5 + diff * gain, 0, 255).astype("uint8")
    mag = np.sqrt((diff ** 2).sum(axis=2))
    mag /= max(float(mag.max()), 1e-6)
    heatmap = (cm.inferno(mag)[:, :, :3] * 255).astype("uint8")
    return signed, heatmap, gain


def add_attack_panels(fig, d, x0=0.36, y0=0.075, w=0.066, gap=0.005, budget=None):
    """Compact clean / GeoFA-TD δ / GeoFA-TD attacked / GeoShield attacked panels, overlaid
    on the map's empty Southern-Ocean band so the figure stays map-width. Returns the δ
    gain (or None if the GeoFA-TD images are missing). No |δ|-heatmap panel here
    (GeoShield's perturbation isn't available to us -- only its attacked image -- so a
    heatmap couldn't be shown for it anyway, and dropping GeoFA-TD's keeps both attacks'
    panels directly comparable)."""
    if not ((d / "clean.png").exists() and (d / "dtd.png").exists()):
        return None
    clean_img = Image.open(d / "clean.png").convert("RGB")
    dtd_img = Image.open(d / "dtd.png").convert("RGB")
    signed, _, gain = perturbation_views(clean_img, dtd_img)
    td = _display_attack_name("dtd")
    panels = [(clean_img, "clean $c$"),
              (signed, td + r" $\delta^{\mathrm{TD}}$" + f" ($\\times${gain:.0f})"),
              (dtd_img, td + r" $c{+}\delta^{\mathrm{TD}}$")]
    geoshield_path = d / "geoshield.png"
    has_geoshield = geoshield_path.exists()
    if has_geoshield:
        panels.append((Image.open(geoshield_path).convert("RGB"),
                       _display_attack_name("geoshield") + r" $c{+}\delta^{\mathrm{GS}}$"))
    h = w * (fig.get_figwidth() / fig.get_figheight())  # keep panels square
    n = len(panels)
    heading = f"Ours ({td}) vs. GeoShield" if has_geoshield else f"{td} (ours)"
    if budget is not None:
        heading += f"  ($\\Vert\\delta\\Vert_\\infty \\leq {round(budget * 255)}/255$)"
    fig.text(x0 + ((n - 1) / 2) * (w + gap), y0 + h + 0.035, heading,
             ha="center", va="bottom", fontsize=9, weight="bold")
    for i, (im, cap) in enumerate(panels):
        axi = fig.add_axes([x0 + i * (w + gap), y0, w, h])
        axi.imshow(im)
        axi.set_xticks([]); axi.set_yticks([])
        axi.set_title(cap, fontsize=7.5, pad=2)
        for s in axi.spines.values():
            s.set_edgecolor("#444444"); s.set_linewidth(0.8)
    return gain

def save_tight(fig, path, pad_inches=0.03, **kwargs):
    """``fig.savefig(path, bbox_inches="tight")`` that survives cartopy's gridliner.

    With ``gl.top_labels = gl.right_labels = False`` cartopy (0.25, matplotlib 3.11) keeps
    hidden label ``Text`` artists whose tight bbox is +-inf, which makes the GeoAxes' bbox
    NaN and silently drops the whole map from the "tight" crop (only the inset and the
    image panels survived). We union the finite bboxes of every axes and artist instead."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    boxes = []
    for ax_ in fig.axes:
        boxes.append(ax_.get_window_extent(r))
        for a in ax_.get_children():
            try:
                bb = a.get_tightbbox(r)
            except Exception:
                continue
            if bb is not None and np.all(np.isfinite(bb.extents)) and bb.width > 0 and bb.height > 0:
                boxes.append(bb)
    boxes += [t.get_window_extent(r) for t in fig.texts]
    bb = Bbox.union(boxes)
    bbox_in = Bbox.from_extents(bb.x0 / fig.dpi, bb.y0 / fig.dpi, bb.x1 / fig.dpi, bb.y1 / fig.dpi)
    fig.savefig(path, bbox_inches=bbox_in.padded(pad_inches), **kwargs)


print("Map helpers defined.")

In [ ]:
# Compute (or load cached) predicted locations for the clean image and every attack.
example_gps_by_id = {iid: tuple(g) for iid, g in zip(example_ids, example_gps)}
predictions = {}  # (image_id, budget) -> {"true":(lat,lon), "clean":[lat,lon], "attacks":{key:[lat,lon]}}
clean_preds = {}  # image_id -> medoid of the clean prediction (shared by every budget)

def _cache_is_fresh(cache, d):
    """The cache is stale if any attack image was (re)generated after it, or if it
    was written for a different attack set (e.g. before the GeoFA renaming)."""
    if not cache.exists():
        return False
    imgs = [d / f"{k}.png" for k in ATTACK_ORDER]  # clean.png is rewritten (identically) each run
    if any(p.exists() and p.stat().st_mtime > cache.stat().st_mtime for p in imgs):
        return False
    cached = set(json.load(open(cache)).get("attacks", {}))
    present = {k for k in ATTACK_ORDER if (d / f"{k}.png").exists()}
    return cached == present

for budget in ATTACK_BUDGETS:
    eps_tag = f"{round(budget * 255)}/255"
    for iid in example_ids:
        d = budget_dir(iid, budget)
        cache = d / "prediction_points.json"
        if _cache_is_fresh(cache, d) and not RECOMPUTE_PREDICTIONS:
            predictions[(iid, budget)] = json.load(open(cache))
            print(f"[{iid} @ {eps_tag}] loaded cached predictions")
            continue
        if not (d / "clean.png").exists():
            print(f"[{iid} @ {eps_tag}] no clean.png; run the attack cells first. Skipping.")
            continue
        if iid not in clean_preds:
            clean_preds[iid] = medoid(predict_gps(Image.open(d / "clean.png").convert("RGB")))
        attacks = {}
        for key in ATTACK_ORDER:
            p = d / f"{key}.png"
            if p.exists():
                attacks[key] = [float(x) for x in medoid(predict_gps(Image.open(p).convert("RGB")))]
        rec = {"true": list(example_gps_by_id[iid]), "budget_over_255": round(budget * 255),
               "clean": [float(x) for x in clean_preds[iid]], "attacks": attacks}
        json.dump(rec, open(cache, "w"), indent=2)
        predictions[(iid, budget)] = rec
        print(f"[{iid} @ {eps_tag}] computed predictions for {len(attacks)} attacks")

In [ ]:
# Draw one publication-ready map per image (global view + exploded zoom inset).
for (iid, budget), rec in predictions.items():
    d = budget_dir(iid, budget)
    eps_tag = f"{round(budget * 255)}/255"
    true_gps = tuple(rec["true"])
    clean_pred = np.asarray(rec["clean"], dtype=float)
    preds = {l: np.asarray(p, dtype=float) for l, p in rec["attacks"].items()}

    fig = plt.figure(figsize=(12, 5.6))
    ax = fig.add_axes([0.025, 0.04, 0.80, 0.93], projection=ccrs.PlateCarree())
    ax.set_global()
    _style_map(ax)
    _draw_points(ax, true_gps, clean_pred, preds, fan_span=GLOBAL_FAN_SPAN, explode_clean=False)

    # zoom inset framing the true/clean cluster (large, lower-left)
    near = [true_gps, tuple(clean_pred)] + [
        tuple(p) for p in preds.values() if haversine_km(tuple(clean_pred), p)[0] < 1500
    ]
    near = np.array(near)
    lat_c, lon_c = float(near[:, 0].mean()), float(near[:, 1].mean())
    span = max(7.0, float(np.ptp(near[:, 0])) * 1.7, float(np.ptp(near[:, 1])) * 1.7)
    ext = [lon_c - span / 2, lon_c + span / 2, lat_c - span / 2, lat_c + span / 2]
    axins = fig.add_axes([0.0, 0.085, 0.26, 0.42], projection=ccrs.PlateCarree())
    axins.set_extent(ext, crs=ccrs.PlateCarree())
    _style_map(axins, labels=False)
    preds_in = {l: p for l, p in preds.items()
                if ext[0] <= p[1] <= ext[1] and ext[2] <= p[0] <= ext[3]}
    _draw_points(axins, true_gps, clean_pred, preds_in, with_connectors=False, fan_span=span)
    for s in axins.spines.values():
        s.set_edgecolor("#444444"); s.set_linewidth(1.0)
    axins.set_title("zoom near true location", fontsize=8)

    # GeoFA-TD (ours) + GeoShield attack panels overlaid on the map's (empty) Southern-Ocean
    # band, so the figure stays map-width rather than growing a side column.
    add_attack_panels(fig, d, x0=0.25, y0=0.085, w=0.11, budget=budget)
    if "dtd" in preds:
        dtd_km = float(haversine_km(tuple(clean_pred), preds["dtd"])[0])
        print(f"[{iid} @ {eps_tag}] GeoFA-TD moves the prediction by {dtd_km:,.0f} km")

    # legend overlaid inside the map's top-right corner (semi-transparent so markers behind
    # it stay readable). No external margin => bbox_inches="tight" trims the figure to the map.
    leg = ax.legend(handles=legend_handles(preds), loc="upper right",
                    fontsize=8.5, frameon=True, framealpha=0.92, facecolor="white",
                    edgecolor="#cccccc", borderpad=0.6, labelspacing=0.35, handletextpad=0.5)
    leg.set_zorder(20)

    save_tight(fig, d / "prediction_map.png", dpi=200)
    # max-quality copy for the paper (vector map, 1200 dpi rasters):
    save_tight(fig, d / "prediction_map_max.pdf", dpi=1200)
    plt.show()
    print("saved", d / "prediction_map.png")

## Save metadata

In [ ]:
metadata = {
    "dataset": DATASET,
    "budgets_over_255": [round(b * 255) for b in ATTACK_BUDGETS],
    "primary_budget_over_255": round(PRIMARY_BUDGET * 255),
    "example_image_ids": EXAMPLE_IMAGE_IDS,
    "selection_seed": SELECTION_SEED,
    "attack_seed": ATTACK_SEED,
    "attack_config": "configs/cluster_full/config.yaml",
    "fast_mode": FAST_MODE,
    "image_ids": example_ids,
    "image_gps": {iid: list(gps) for iid, gps in zip(example_ids, example_gps)},
    "attacks": list(ATTACKS) + (["geoshield"] if RUN_GEOSHIELD else []),
    "attack_display_names": {k: _display_attack_name(k)
                             for k in list(ATTACKS) + (["geoshield"] if RUN_GEOSHIELD else [])},
    "records": records,
}
with open(OUTPUT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Wrote", (OUTPUT_DIR / "metadata.json").resolve())
print("Example folders:")
for iid in example_ids:
    print("  ", image_dir(iid).resolve())

## TikZ teaser figure (paper hero figure)

A standalone, hand-editable pgfplots twin of the map above, built specifically for the
paper's teaser: instead of every attack, it pins just the **source photo** (at its true
location) alongside the **GeoFA-TD (ours)** and **GeoShield** attacked photos (at their
predicted locations), connected by great-circle lines back to the clean prediction --
same story as the map above, fewer attacks, real image thumbnails as TikZ nodes.

Reads the same cached `prediction_points.json` (no re-running the pipeline). Outputs
land in `<budget_dir>/tikz/teaser_<image_id>.{tex,pdf,png}` (one per image and budget) (+ copies of the three PNGs
and the background world map next to the `.tex`, so that folder is self-contained and
portable). The `.pdf`/`.png` are a best-effort local compile (via pdflatex, falling back
to lualatex for large data) so you can preview it here; the `.tex` itself will compile
anywhere pgfplots is available (e.g. Overleaf) and every marker/line/label is a normal
TikZ node you can nudge by hand -- e.g. if the legend or a thumbnail happens to land
somewhere awkward for a specific image.

Pick whichever `example_ids` entry tells the best story for the teaser (biggest
GeoFA-TD/GeoShield displacement, clearest map layout, etc.) -- there's one figure per image.

In [ ]:
from utils.plots_tikz import plot_geolocation_teaser_tikz

teaser_results = {}
for budget in ATTACK_BUDGETS:
    for iid in example_ids:
        d = budget_dir(iid, budget)
        if not (d / "prediction_points.json").exists():
            print(f"[{iid} @ {round(budget * 255)}/255] no prediction_points.json; run the map cells above first. Skipping.")
            continue
        stem = f"teaser_{iid}" if d == image_dir(iid) else f"teaser_{iid}_{budget_label(budget)}"
        teaser_results[(iid, budget)] = plot_geolocation_teaser_tikz(
            image_dir=d, attack_labels=("dtd", "geoshield"), stem=stem)

print("\nDone. Preview PNGs (if pdflatex/lualatex are available):")
for (iid, budget), r in teaser_results.items():
    print(f"  {iid} @ {round(budget * 255)}/255: {r.get('png') or r['tex']}")